# Week 3: LoRA + replay

Trains the proposed method and its two ablations over P1 to P4 on a Kaggle GPU, 3 seeds each,
and logs every run to MLflow on DagsHub. See `docs/design.md` sections 7 and 8,
`src/training/run.py` and `docs/baselines.md` for the baselines this is compared with.

| Run | What it shows |
| --- | --- |
| `lora_replay` with 1%, 5% and 10% replay | the proposed method, and how the buffer size matters |
| `lora` | LoRA alone: what the adapters do without replay |
| `replay` with 5% replay | normal fine-tuning plus replay: what replay does without LoRA |

15 runs in total. Expected time: about 75 minutes (an estimate, not measured).

**Before you run**

1. Settings: Accelerator **GPU T4 x2**, Internet **on**.
2. Add-ons, Secrets: attach `DAGSHUB_TOKEN` (the same token as `.dvc/config.local`). Secrets
   are not kept when a notebook is pushed with the Kaggle CLI, so attach it again after every
   CLI push.
3. Choose **Save & Run All (Commit)**. It runs in the background and the GPU session ends by
   itself when the notebook finishes or fails. Do not use an interactive session: it keeps the
   GPU until you stop it.

The token is only put into environment variables and is never printed. The run stops at the
first error so no GPU time is wasted. At the end the cloned repository, with its data, is
deleted, so the notebook output holds only the result files and one adapter.

## 1. Credentials

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

# DVC reads its S3 keys from these variables, and MLflow reads the other three.
os.environ["AWS_ACCESS_KEY_ID"] = token
os.environ["AWS_SECRET_ACCESS_KEY"] = token
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/Akmal76/slangguard-data.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "Akmal76"
os.environ["MLFLOW_TRACKING_PASSWORD"] = token

## 2. Code and data

Kaggle already has torch and transformers. `peft` provides LoRA, `mlflow-skinny` is the small
MLflow client (enough to log to a server), `dvc[s3]` pulls the data, and `pytest` runs the
check below.

The Kaggle image ships `torchao` 0.10.0, which `peft` rejects as too old when it adds LoRA
to a model. This project does not use `torchao`, so it is uninstalled.

In [ ]:
COMMIT = "ddfbd1a"  # the code of this run. It must exist on GitHub.

!git clone -q https://github.com/Akmal76/slangguard.git
%cd slangguard
!git checkout -q {COMMIT}
!git log -1 --format="code at commit %h: %s"

In [ ]:
!pip uninstall -y -q torchao
!pip install -q peft==0.21.2 mlflow-skinny==3.18.0 'dvc[s3]' pytest

In [ ]:
%%bash
set -e
# LoRA tests on a tiny random model: they need no data or GPU and use the library versions of
# this machine, so a version clash fails here in seconds instead of in the middle of training.
python -m pytest -q tests/test_train.py -k lora

In [ ]:
!dvc pull data/periods

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Turn on a GPU accelerator in the notebook settings.")
print(torch.cuda.get_device_name(0))

## 3. Train

Seeds are the outer loop, so if the run stops early the finished seeds are complete.
`set -e` stops everything at the first error. Run names in MLflow: `lora_replay-r1-seed42`,
`lora-seed42`, `replay-r5-seed42` and so on.

In [ ]:
%%bash
set -e
for seed in 42 43 44; do
    for replay in 0.01 0.05 0.10; do
        python -m src.training.run --method lora_replay --replay $replay --seed $seed
    done
    python -m src.training.run --method lora --seed $seed
    python -m src.training.run --method replay --replay 0.05 --seed $seed
done

## 4. Results

The result files go to `/kaggle/working/results`, and the adapter that will be published
(`lora_replay` with 5% replay, seed 42, chosen before any result was seen) goes to
`/kaggle/working/adapters`. Then the cloned repository and its data are deleted.

In [ ]:
import json
import shutil
from pathlib import Path

repo = Path.cwd()
results = Path("/kaggle/working/results")
results.mkdir(exist_ok=True)

for path in sorted(Path("outputs").glob("*.json")):
    r = json.loads(path.read_text())
    print(
        f"{r['run']:24s} AvgF1 {r['avg_f1']:.3f}  forgetting {r['forgetting']:.3f}  "
        f"P1 after step 4 {r['matrix'][-1][0]:.3f}  {r['total_train_seconds'] / 60:.1f} min"
    )
    shutil.copy(path, results)

adapter = "lora_replay-r5-seed42"
shutil.copytree(Path("outputs/adapters") / adapter, Path("/kaggle/working/adapters") / adapter)

%cd /kaggle/working
shutil.rmtree(repo)
print(sorted(p.name for p in Path("/kaggle/working").iterdir()))